# Project: Identify Customer Segments

In this project, you will apply unsupervised learning techniques to identify segments of the population that form the core customer base for a mail-order sales company in Germany. These segments can then be used to direct marketing campaigns towards audiences that will have the highest expected rate of returns. The data that you will use has been provided by our partners at Bertelsmann Arvato Analytics, and represents a real-life data science task.

This notebook will help you complete this task by providing a framework within which you will perform your analysis steps. In each step of the project, you will see some text describing the subtask that you will perform, followed by one or more code cells for you to complete your work. **Feel free to add additional code and markdown cells as you go along so that you can explore everything in precise chunks.** The code cells provided in the base template will outline only the major tasks, and will usually not be enough to cover all of the minor tasks that comprise it.

It should be noted that while there will be precise guidelines on how you should handle certain tasks in the project, there will also be places where an exact specification is not provided. **There will be times in the project where you will need to make and justify your own decisions on how to treat the data.** These are places where there may not be only one way to handle the data. In real-life tasks, there may be many valid ways to approach an analysis task. One of the most important things you can do is clearly document your approach so that other scientists can understand the decisions you've made.

At the end of most sections, there will be a Markdown cell labeled **Discussion**. In these cells, you will report your findings for the completed section, as well as document the decisions that you made in your approach to each subtask. **Your project will be evaluated not just on the code used to complete the tasks outlined, but also your communication about your observations and conclusions at each stage.**

In [ ]:
# import libraries here; add more as necessary
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import anglit

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from pprint import pp as pprint

# magic word for producing visualizations in notebook
%matplotlib inline

pd.options.display.max_rows = None
pd.options.display.max_columns = None

In [ ]:
# CONSTANTS FOR FEATURE TYPES
BINARY = "binary_categorical"
MULTI = "multi_categorical"
NUMERICAL = "numerical_features"
NON_NUM = "non_numerical_categorical"
ORDINAL = "ordinal_features"
INTERVAL = "interval_features"
ENGINEERED = "engineered"
TO_ENGINEER = "to_engineer"
TO_DROP = "features_to_drop"
TO_KEEP = "features_to_keep"

Learn FunctionTransformer and Pipeline: https://www.kaggle.com/code/gamirandads/learn-functiontransformer-and-pipeline

### Step 0: Load the Data

There are four files associated with this project (not including this one):

- `Udacity_AZDIAS_Subset.csv`: Demographics data for the general population of Germany; 891211 persons (rows) x 85 features (columns).
- `Udacity_CUSTOMERS_Subset.csv`: Demographics data for customers of a mail-order company; 191652 persons (rows) x 85 features (columns).
- `Data_Dictionary.md`: Detailed information file about the features in the provided datasets.
- `AZDIAS_Feature_Summary.csv`: Summary of feature attributes for demographics data; 85 features (rows) x 4 columns

Each row of the demographics files represents a single person, but also includes information outside of individuals, including information about their household, building, and neighborhood. You will use this information to cluster the general population into groups with similar demographic properties. Then, you will see how the people in the customers dataset fit into those created clusters. The hope here is that certain clusters are over-represented in the customers data, as compared to the general population; those over-represented clusters will be assumed to be part of the core userbase. This information can then be used for further applications, such as targeting for a marketing campaign.

To start off with, load in the demographics data for the general population into a pandas DataFrame, and do the same for the feature attributes summary. Note for all of the `.csv` data files in this project: they're semicolon (`;`) delimited, so you'll need an additional argument in your [`read_csv()`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_csv.html) call to read in the data properly. Also, considering the size of the main dataset, it may take some time for it to load completely.

Once the dataset is loaded, it's recommended that you take a little bit of time just browsing the general structure of the dataset and feature summary file. You'll be getting deep into the innards of the cleaning in the first major step of the project, so gaining some general familiarity can help you get your bearings.

In [ ]:
def load_data():
    # Load in the general demographics' data.
    # I have named it temp, as I will be using it for testing and then
    # loading the data again for the actual training and analysis
    df = pd.read_csv("Udacity_AZDIAS_Subset.csv", delimiter=";")

    # Load in the feature summary file.
    fs = pd.read_csv("AZDIAS_Feature_Summary.csv", delimiter=";")

    return df, fs

azidas_raw, feature_summary = load_data()

#### Exploring the General Population Data:

In [ ]:
# Print the first 5 rows
azidas_raw.head()

In [ ]:
# Get some more information about the data
azidas_raw.info()

In [ ]:
azidas_raw.columns.to_list()

In [ ]:
# Use .describe() to get summary statistics
azidas_raw.describe()

In [ ]:
# Get a peek of the data types
azidas_raw.dtypes.value_counts()

So basic information about the general population demographics data:
- There are 891221 entries/samples, or rows, each representing one person
- There are a total of 85 rows or features
- There are 81 numeric features (32xint64 + 49xfloat64) and 4 non-numeric features (objects)

Now we explore how many NaN values are included in the data:

In [ ]:
# Calculate the total number of NaN values in the dataset
print(f"Number of NaN values in demographics dataset: {azidas_raw.isna().sum().sum()}")

# Get the count of NaN values per column or feature
azidas_raw.isna().sum().sort_values(ascending=False)

As we can see, several features have NaN values, totaling 4,896,838. **NOTE**: this are actual NaN values recorded in the data. There could still be missings or unknowns in the data.

#### Exploring Feature Summary Data:

In [ ]:
feature_summary.head()

In [ ]:
feature_summary.info()

In [ ]:
# Explore the "information_level" column. Get the values and counts
feature_summary["information_level"].value_counts()

In [ ]:
# Explore the "type" column. Get the values and counts
feature_summary["type"].value_counts()

The feature summary basically gives us the following information about each attribute/column/feature:
- attribute: the name of the attribute/feature
- information_level: attributes can be personal or higher-level (ex. geographical)
- type: the kind of data that an attribute or feature contains: orinal (ranked), categorical (labels/codes), numeric (numbers), mixed or interval
- missing_or_unknown: values for missing or unknown attribute data

#### Loading Data Dictionary:

I decided to load the Data Dictionary from a JSON file so that I can reference it in this notebook.

In [ ]:
import json
# Load Data Dictionary
with open("data_dictionary.json") as f:
    data_dictionary = json.load(f)

print(len(data_dictionary))
pprint(data_dictionary, depth=1)

> **Tip**: Add additional cells to keep everything in reasonably-sized chunks! Keyboard shortcut `esc --> a` (press escape to enter command mode, then press the 'A' key) adds a new cell before the active cell, and `esc --> b` adds a new cell after the active cell. If you need to convert an active cell to a markdown cell, use `esc --> m` and to convert to a code cell, use `esc --> y`. 

## Step 1: Preprocessing

### Step 1.1: Assess Missing Data

The feature summary file contains a summary of properties for each demographics data column. You will use this file to help you make cleaning decisions during this stage of the project. First of all, you should assess the demographics data in terms of missing data. Pay attention to the following points as you perform your analysis, and take notes on what you observe. Make sure that you fill in the **Discussion** cell with your findings and decisions at the end of each step that has one!

#### Step 1.1.1: Convert Missing Value Codes to NaNs
The fourth column of the feature attributes summary (loaded in above as `feat_info`) documents the codes from the data dictionary that indicate missing or unknown data. While the file encodes this as a list (e.g. `[-1,0]`), this will get read in as a string object. You'll need to do a little bit of parsing to make use of it to identify and clean the data. Convert data that matches a 'missing' or 'unknown' value code into a numpy NaN value. You might want to see how much data takes on a 'missing' or 'unknown' code, and how much data is naturally missing, as a point of interest.

**As one more reminder, you are encouraged to add additional cells to break up your analysis into manageable chunks.**

In [ ]:
# Real NaN values
real_na_total = azidas_raw.isnull().sum().sum()
print(f"Total Real Missing Data: {real_na_total}")

#### Identifying Missing Data:

In [ ]:
# Identify missing or unknown data values and convert them to NaNs.
for attribute, missing_encoded_str in zip(feature_summary["attribute"], feature_summary["missing_or_unknown"]):
    print(f"Feature {attribute} encodes missing values as: {missing_encoded_str}, {type(missing_encoded_str)}")

Convert missing_or_unknown strings into actual lists of values:

In [ ]:
def get_missing_summary(fs):
    """
    Function that returns a dictionary mapping each attribute in a feature summary
    To a list of the missing or unknown codes for that attribute
    :param fs: Feature summary
    :return: Dictionary mapping attribute to a list of missing or unknown codes
    """

    def get_numeric_value(s):
        """
        Basic function that attempts to cast a string into an int.
        :param s: the string to cast to int
        :return: the integer represented by the string or the original string if invalid string
        """
        try:
            return int(s)
        except ValueError:
            return s

    # Init dictionary
    missing_summary = {}

    # iterate over attributes (to use as key) and missing_or_unknown values (as strings now)
    for attribute, missing_code_str in zip(fs["attribute"], fs["missing_or_unknown"]):

        # clean up the string and create a list
        temp_list = list(missing_code_str.replace(" ", "").replace("[","").replace("]","").split(","))

        # check that the list is not empty
        if len(temp_list) > 0 and temp_list[0] != "":
            # convert strings into numerical values
            missing_summary[attribute] = [get_numeric_value(s) for s in temp_list]

    return missing_summary

print(f"miss_or_unkwn dictionary: {get_missing_summary(feature_summary)}")

Now we go over all columns having missing or unknown values and replace them with NaN:

In [ ]:
def replace_missing(df, fs, return_count=False, verbose=False):
    """
    Replaces missing or unknown coded values in DataFrame with NaN
    :param df: Target DataFrame
    :param fs: Feature Summary
    :param return_count: flag for if we want to return the count of replaced values
    :param verbose: flag for if we want to print more information
    :return: tuple(converted DataFrame, count of replaced values)
    or just the converted DataFrame, depending on return_count flag
    """
    # Make a copy
    df = df.copy()

    missing_summary = get_missing_summary(fs)

    # keep track of total missing or unknown datums (so we can have a separate value for real NaN values in the data
    # and converted NaN values). Used later for sanity check.
    converted_na_total = 0

    # we iterate over each attribute and list pair in the dictionary
    # we don't iterate over each column, as not all of them have missing_or_unknown codes
    for attribute, lst in missing_summary.items():

        if attribute not in df:
            continue

        # Get the sum of missing or unknown values for that attribute/feature in the general pop. data
        # We use isin() function to match with the list of missing_or_unkown values and then sum them up
        n_miss = df[attribute].isin(lst).sum()

        # Get the count of real NaN values in the dataset for that feature
        n_nan = df[attribute].isna().sum()

        if verbose:
            print(f"{attribute} has {n_miss} missing or unknowns and {n_nan} real NaN values.")

        # Replace all the values with NaN
        df[attribute] = df[attribute].replace(lst, np.nan)

        # increment total
        converted_na_total += n_miss

    return (df, converted_na_total) if return_count else df

# Add to transformers dictionary
replacer_transformer = FunctionTransformer(
    replace_missing,
    kw_args={"fs": feature_summary},
    validate=False,
    feature_names_out="one-to-one"
)

In [ ]:
azidas_nan, converted_na_total = replace_missing(azidas_raw, feature_summary, return_count=True, verbose=True)
print()
print(f"Total number of missing values: {converted_na_total}")

**Sanity Check**: checking that we correctly converted all NaN values in columns

In [ ]:
# Sum of converted NaN values and real NaN values
nan_total = converted_na_total + real_na_total
print(nan_total)

In [ ]:
# If all is good, then the whole dataset should have the same total number as above
print(f"Number of NaN values in general population dataset: {azidas_nan.isna().sum().sum()}")
assert nan_total == azidas_nan.isna().sum().sum()

# all good, output a summary of the NaN values per feature
azidas_nan.isna().sum().sort_values(ascending=False)

#### Step 1.1.2: Assess Missing Data in Each Column

How much missing data is present in each column? There are a few columns that are outliers in terms of the proportion of values that are missing. You will want to use matplotlib's [`hist()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html) function to visualize the distribution of missing value counts to find these columns. Identify and document these columns. While some of these columns might have justifications for keeping or re-encoding the data, for this project you should just remove them from the dataframe. (Feel free to make remarks about these outlier columns in the discussion, however!)

For the remaining features, are there any patterns in which columns have, or share, missing data?

In [ ]:
# Perform an assessment of how much missing data there is in each column of the
# dataset.
col_missing_counts = azidas_nan.isna().sum()
print(col_missing_counts)

In [1]:
# Plot Counts
plt.figure(figsize=(10,6))
plt.bar(col_missing_counts.index, col_missing_counts.values, color="b", edgecolor='black')
plt.title('Missing Values per Feature/Column')
plt.xlabel('Feature/Column')
plt.xticks(rotation=90, ha="center", fontsize=6)
plt.ylabel('Number of NaN values')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Convert to percentages
missing_pct = (col_missing_counts / len(azidas_nan)) * 100
missing_pct.sort_values(ascending=False)

In [ ]:
# Plot Percentages
plt.figure(figsize=(10,6))
plt.bar(missing_pct.index, missing_pct.values, color="r", edgecolor='black')
plt.title('Percentage of Missing Values per Feature/Column')
plt.xlabel('Feature/Column')
plt.xticks(rotation=90, ha="center", fontsize=6)
plt.ylabel('% of Missing (NaN) Values')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

As can be seen in the bar graph, a possible good threshold would be 20% of missing values

In [ ]:
# Use pd.cut() to separate each column's percentage of missing values into defined ranges (bins)
bins = [0, 10, 20, 30, 40, 50,75, 100]
labels = ["0–10%", "10–20%", "20-30%", "30-40%", "40–50%",'50-75%', '75–100%']
missing_pct_bins = pd.cut(missing_pct, bins=bins, labels=labels, include_lowest=True)

# Summarize by getting the count of each value and sorting
print(missing_pct_bins.value_counts().sort_index())

As can be seen, out of the 85 features/columns, 79 columns have 20% or less missing values, and 6 columns have more than 20%.

In [ ]:
# Remove the outlier columns from the dataset. (You'll perform other data
# engineering tasks such as re-encoding and imputation later.)

# Get a list of the columns that have > 20% missing values
columns_to_drop = list(missing_pct[missing_pct > 20].index)
print(f"Columns to Drop: {columns_to_drop}")

In [ ]:
# Feature Summary for the columns that we will drop
drop_summary = feature_summary[feature_summary["attribute"].isin(columns_to_drop)]
drop_summary

#### Discussion 1.1.2: Assess Missing Data in Each Column

As can be seen above from the graph and the summary of percentage missing values, the number of missing values is not uniformly distributed across the dataset. 39/85 columns are missing 0-10% and 40/85 columns are missing 10-20% of values. This can be remedied later with imputation. I decided to use 20% missing values as a threshold based on the information gathered.

The columns dropped are:
- **'AGER_TYP'**: Best-ager typology. >70% missing data. Not useful.
- **'GEBURTSJAHR'**: Year of birth. >40% missing data. Not useful.
- **'TITEL_KZ'**: Academic title flag. >99% missing data.
- **'ALTER_HH'**: Birthdate of head of household. >30% missing data. Not useful.
- **'KK_KUNDENTYP'**: Consumer pattern over past 12 months. >65% missing data. Not useful.
- **'KBA05_BAUMAX'**: Most common building type within microcell. >50% missing data. Not useful and probably irrelevant.

#### Step 1.1.3: Assess Missing Data in Each Row

Now, you'll perform a similar assessment for the rows of the dataset. How much data is missing in each row? As with the columns, you should see some groups of points that have a very different numbers of missing values. Divide the data into two subsets: one for data points that are above some threshold for missing values, and a second subset for points below that threshold.

In order to know what to do with the outlier rows, we should see if the distribution of data values on columns that are not missing data (or are missing very little data) are similar or different between the two groups. Select at least five of these columns and compare the distribution of values.
- You can use seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) function to create a bar chart of code frequencies and matplotlib's [`subplot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplot.html) function to put bar charts for the two subplots side by side.
- To reduce repeated code, you might want to write a function that can perform this comparison, taking as one of its arguments a column to be compared.

Depending on what you observe in your comparison, this will have implications on how you approach your conclusions later in the analysis. If the distributions of non-missing features look similar between the data with many missing values and the data with few or no missing values, then we could argue that simply dropping those points from the analysis won't present a major issue. On the other hand, if the data with many missing values looks very different from the data with few or no missing values, then we should make a note on those data as special. We'll revisit these data later on. **Either way, you should continue your analysis for now using just the subset of the data with few or no missing values.**

In [ ]:
def get_na_per_row(df):
    # How much data is missing in each row of the dataset?
    # Get the sum of NaN values per row
    na_p_row = df.isna().sum(axis=1)
    print(na_p_row.describe())
    return na_p_row

na_per_row = get_na_per_row(azidas_nan)

In [ ]:
# For convenience, we turn into a Data Frame
na_per_row_df = pd.DataFrame(na_per_row, columns=["# of NaN values"])
na_per_row_df.head(20)

In [ ]:
# Plot a histogram
plt.hist(na_per_row, bins=50);
plt.xlabel('Number of NaN Values')
plt.ylabel('Number of Rows')
plt.title('Distribution of Missing Values per Row', fontweight='bold')
plt.show()

As can be seen by the histogram, most of the rows are complete, with a small spike of rows missing

In [ ]:
# Plot again, focusing on rows that have <= 30 missing values
plt.hist(na_per_row[na_per_row <= 30], bins=20)
plt.xlabel('Number of Missing Values')
plt.ylabel('Number of Rows')
plt.title('Distribution of Missing Values per Row for Rows Missing 20 or less Values', fontweight='bold')
plt.show()

In [ ]:
class DropMissingTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, threshold, axis):
        self.threshold = threshold
        self.axis = axis

        self._dropped_columns = []
        self._columns = None

        self._dropped_row_count = 0

        self._high_na_df = None
        self._low_na_df = None

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X = X.copy()

        # check for axis
        if self.axis == 0:
            # drop rows
            print(f"Dropping Rows based on threshold: {self.threshold}")

            # spilt into two subsets based on number of NaN values per row
            high_na_df = X[X.isna().sum(axis=1) > self.threshold]
            low_na_df = X[X.isna().sum(axis=1) <= self.threshold]

            self._high_na_df = high_na_df
            self._low_na_df = low_na_df
            self._dropped_row_count = len(X) - len(low_na_df)

            self._columns = X.columns.to_list()

            return low_na_df

        else:
            print(f"Dropping Columns based on threshold: {self.threshold}")
             # get NaN count
            na_count = X.isna().sum()
            # get percentages and sort
            na_pct = (na_count / len(X)) * 100

            # get list of columns to drop, based on those that have a percentage of missings > than the threshold
            cols_to_drop = list(na_pct[na_pct > self.threshold].index)

            dropped_columns = []

            for col in cols_to_drop:
                if col in X.columns:
                    try:
                        # if so, drop them
                        X.drop(col, axis="columns", inplace=True)
                    except KeyError:
                        # error
                        print(f"Could not find {col}")
                    else:
                        dropped_columns.append(col)

            self._dropped_columns = dropped_columns
            self._columns = X.columns.to_list()
            # no columns to drop? then simply return df
            return X

    def get_feature_names_out(self, input_features=None):
        return self._columns

    def get_dropped_columns(self):
        return self._dropped_columns

    def get_dropped_row_count(self):
        return self._dropped_row_count

    def get_data_subsets(self):
        return self._low_na_df, self._high_na_df

In [ ]:
# def drop_missing_threshold(df, threshold, axis, return_both=False):
#     """
#     Function that drops rows or columns from a DataFrame, based on a threshold
#     :param df: DataFrame to drop rows or columns
#     :param threshold: threshold to decide
#     :param axis: 0 for rows, 1 for columns
#     :param return_both: flag for if we want to return both high and low subsets after splitting (for axis=0)
#     by rows.
#     :return:    if axis = 0, returns either both subsets of high and low missings if return_both, else just the high missings subset
#                 if axis = 1, returns the df with columns that were dropped (if any)
#     """
#
#     # check for axis
#     if axis == 0:
#
#         # spilt into two subsets based on number of NaN values per row
#         high_na_df = df[df.isna().sum(axis=1) > threshold]
#         low_na_df = df[df.isna().sum(axis=1) <= threshold]
#
#         return (low_na_df, high_na_df) if return_both else low_na_df
#     # else, dropping rows
#     else:
#         # get NaN count
#         na_count = df.isna().sum()
#         # get percentages and sort
#         na_pct = (na_count / len(df)) * 100
#
#         # get list of columns to drop, based on those that have a percentage of missings > than the threshold
#         cols_to_drop = list(na_pct[na_pct > threshold].index)
#
#         for col in cols_to_drop:
#             try:
#                 # check that all the columns we will drop are in the df
#                 if col in df.columns:
#                     # if so, drop them
#                     df.drop(col, axis="columns", inplace=True)
#             except KeyError:
#                 # error
#                 print(f"Could not find {cols_to_drop}")
#             return df
#         else:
#             # no columns to drop? then simply return df
#             return df
#
# transformers["drop_na_cols"] = FunctionTransformer(
#     drop_missing_threshold,
#     kw_args= {"threshold": 20, "axis": 1, "return_both":False},
#     validate=False,
# )
#
# transformers["drop_na_rows"] = FunctionTransformer(
#     drop_missing_threshold,
#     kw_args= {"threshold": 20, "axis": 0},
#     validate=False,
#     feature_names_out="one-to-one"
# )

In [ ]:
# Drop columns
na_col_dropper = DropMissingTransformer(threshold=20, axis=1)
azidas_cols_clean = na_col_dropper.transform(azidas_nan)

print(f"Columns successfully dropped: {na_col_dropper.get_dropped_columns()}")

In [ ]:
features_to_drop = {}
features_to_drop["na_cols"] = na_col_dropper.get_dropped_columns()

In [ ]:
azidas_cols_clean.head()

In [ ]:
# Drop rows
na_row_dropper = DropMissingTransformer(threshold=20, axis=0)
na_row_dropper.transform(azidas_cols_clean)

# Separate into two subsets, accept and reject, based on the threshold of 20 missing values per row
azdias_low_missing, azdias_high_missing = na_row_dropper.get_data_subsets()
print(f"Number of rows dropped: {na_row_dropper.get_dropped_row_count()}")

In [ ]:
azdias_low_missing.head()

In [ ]:
print(f"Total # of rows after dropping columns: {len(azidas_cols_clean)}")
print(f"Total # of missing values in rows: {na_per_row_df.sum().item()}")
print(f"Total # of rows in low missing subset: {len(azdias_low_missing)}")
print(f"Total # of missing values in rows of low missing subset: {azdias_low_missing.isna().sum().sum()}")

We have therefore decreased the number of rows from 891,221 to 797,426 (\~10% decrease) and the total number of missing values from 5,035,304 to 999,462 (\~80% decrease).

#### Comparing Distribution of Values Between Accepted and Rejected Subsets

In [ ]:
import random
random.seed(42)

# Select 5 features/columns that have 0 missing values
# First we get all columns with 0 missing values
zero_missing_cols = missing_pct[missing_pct == 0].index.to_list()

# Then we choose 5 random ones
comparison_cols = random.choices(zero_missing_cols, k=5)
print(f"Columns/Features that will be compared: {comparison_cols}")

In [ ]:
# Compare the distribution of values for at least five columns where there are
# no or few missing values, between the two subsets.
def plot_subset_missing_dist(col_label):
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_figwidth(15)

    sns.countplot(x=col_label, data=azdias_low_missing, ax=ax1)
    sns.countplot(x=col_label, data=azdias_high_missing, ax=ax2)

    plt.show()

In [ ]:
for c in comparison_cols:
    plot_subset_missing_dist(c)

#### Discussion 1.1.3: Assess Missing Data in Each Row

In [ ]:
get_na_per_row(azidas_cols_clean);



As seen in the cells above, for na_per_row, the mean is ~5.64, so ~6 features are missing per row, which I would say is rather acceptable. 50% of the data has complete rows, and 75% is missing only 3 values. STD is 13.23, which is rather high. What we observe then is that there are rows that have many missing values (up to 49 max), which are simply unreliable and should be dropped (or not used, as per our subset data).

Comparison of the plots shows that the distributions between accepted and rejected are different, so we cannot simply drop those rows. However, despite the fact that we cannot drop the rows, we can notice on the rejected subset that some features have biased distributions since we can see bars on the chart that are dominated by a single value/category.

### Step 1.2: Select and Re-Encode Features

Checking for missing data isn't the only way in which you can prepare a dataset for analysis. Since the unsupervised learning techniques to be used will only work on data that is encoded numerically, you need to make a few encoding changes or additional assumptions to be able to make progress. In addition, while almost all of the values in the dataset are encoded using numbers, not all of them represent numeric values. Check the third column of the feature summary (`feat_info`) for a summary of types of measurement.
- For numeric and interval data, these features can be kept without changes.
- Most of the variables in the dataset are ordinal in nature. While ordinal values may technically be non-linear in spacing, make the simplifying assumption that the ordinal variables can be treated as being interval in nature (that is, kept without any changes).
- Special handling may be necessary for the remaining two variable types: categorical, and 'mixed'.

In the first two parts of this sub-step, you will perform an investigation of the categorical and mixed-type features and make a decision on each of them, whether you will keep, drop, or re-encode each. Then, in the last part, you will create a new data frame with only the selected and engineered columns.

Data wrangling is often the trickiest part of the data analysis process, and there's a lot of it to be done here. But stick with it: once you're done with this step, you'll be ready to get to the machine learning parts of the project!

In [ ]:
# How many features are there of each data type?

# get a list of features in our pop_accepted_tmp data
feature_list = azdias_low_missing.columns.to_list()

# Index only those attributes that are in our gen_accept_subset
feature_summary[feature_summary["attribute"].isin(feature_list)]["type"].value_counts()

#### Step 1.2.1: Re-Encode Categorical Features

For categorical data, you would ordinarily need to encode the levels as dummy variables. Depending on the number of categories, perform one of the following:
- For binary (two-level) categoricals that take numeric values, you can keep them without needing to do anything.
- There is one binary variable that takes on non-numeric values. For this one, you need to re-encode the values as numbers or create a dummy variable.
- For multi-level categoricals (three or more values), you can choose to encode the values using multiple dummy variables (e.g. via [OneHotEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)), or (to keep things straightforward) just drop them from the analysis. As always, document your choices in the Discussion section.

In [ ]:
# Assess categorical variables: which are binary, which are multi-level, and
# which one needs to be re-encoded?

# Get a list of all the categorical features
categorical_feats = list(feature_summary[
                     (feature_summary["attribute"].isin(feature_list)) &
                     (feature_summary["type"] == "categorical")]["attribute"])

for f in categorical_feats:
    print(f)
    pprint(data_dictionary[f])
    print(f"Unique Values: {azdias_low_missing[f].nunique()} (after replacing missing or unknown values)")
    print()

#### Summary of multi-level categoricals:

- **ANREDE_KZ**: Gender. Could be useful. We keep.
- **CJT_GESAMTTYP**: Customer-Journey-Typology: preferred information and buying channels for consumer. Relevant, keep.
- **FINANZTYP**: Most descriptive financial type for individual. Relevant, keep.
- **GFK_URLAUBERTYP**: Vacation habits. Could be relevant, keep.
- **GREEN_AVANTGARDE**: Membership in environmental sustainability as part of youth. Not really useful for marketing segmentation, drop.
- **LP_FAMILIE_FEIN**: Family type, fine scale. Drop, we will use the rough scale to avoid OneHotEncoding into too many new features.
- **LP_FAMILIE_GROB**: Family type, rough scale. Keep.
- **LP_STATUS_FEIN**: Social status, fine scale. Drop, we will use the rough scale to avoid OneHotEncoding into too many new features.
- **LP_STATUS_GROB**: Social status, rough scale Keep.
- **NATIONALITAET_KZ**: Nationality based on given name analysis. Very broad and few categories. Will drop.
- **SHOPPER_TYP**: Shopper typology. Relevant, keep.
- **SOHO_KZ**: Small office / home office flag. Don't think it is too relevant. I think the other features can capture better information. Drop.
- **VERS_TYP**: Insurance typology. Could be relevant, keep.
- **ZABEOTYP**: Energy consumption typology. Could be relevant, keep.
- **KK_KUNDENTYP**: Consumer pattern over past 12 months. Definitely keep.
- **GEBAEUDETYP**: Type of building (residential vs. commercial). Will drop because I don't think it will capture useful information.
- **OST_WEST_KZ**: Building location via former East / West Germany. I don't know if former East/West Germany can influence consumer behavior. Will keep.
- **CAMEO_DEUG_2015**: German CAMEO: Wealth / Life Stage Typology, rough scale. Relevant, keep.
- **CAMEO_DEU_2015**: German CAMEO: Wealth / Life Stage Typology, detailed scale. Many categories which can lead to "dimentionality course". Drop

In [ ]:
# Define the categoricals we will drop, by choice
features_to_drop["categorical"] = ["GREEN_AVANTGARDE", "LP_FAMILIE_FEIN", "LP_STATUS_FEIN", "NATIONALITAET_KZ", "SOHO_KZ", "GEBAEUDETYP", "CAMEO_DEU_2015"]
features_to_drop["categorical"]

Next, we look at the mixed features:

In [ ]:
# Explore the mixed ones:
# Get a list of all the categorical features
mixed_features = list(feature_summary[
                          (feature_summary["attribute"].isin(feature_list)) &
                          (feature_summary["type"] == "mixed")]["attribute"])

for f in mixed_features:
    print(f)
    pprint(data_dictionary[f])
    print(f"Unique Values: {azdias_low_missing[f].nunique()}")
    print()

#### Summary of Mixed Features:

- **LP_LEBENPHASE_FEIN** and **LP_LEBENPHASE_GROB**: Life stage (fine and rough). These are hard to split into separate features. We will also be creating a life stage feature from "CAMEO_INTL_2025", so we can drop as information could be redundant.

- **PRAEGENDE_JUGENDJAHRE**: We deal with this one further below.

- **WOHNLAGE**: Drop

- **CAMEO_INTL_2015**: We deal with this one further below.

- **PLZ8_BAUMAX**: Drop

In [ ]:
features_to_drop["mixed"] = ["LP_LEBENSPHASE_FEIN", "LP_LEBENSPHASE_GROB", "WOHNLAGE", "PLZ8_BAUMAX"]
features_to_drop["mixed"]

In [ ]:
# init lists to store features based on the type
binary_cats = [] # For binary categorical features
multi_cats = [] # For multi-level categorical

# Iterate over each categorical feature
for cat in categorical_feats:
    # Make sure the feature column still exists in the data we will use
    if cat in azdias_low_missing.columns:
        # Print some helpful information
        print(f"{cat}, unique values: {azdias_low_missing[cat].unique()}, # unique: {azdias_low_missing[cat].nunique()}")

        # Check if binary (two-level) categorical
        if azdias_low_missing[cat].nunique() == 2:
            # append to list
            binary_cats.append(cat)
        # check for multi-level categorical, the ones we will not drop
        elif azdias_low_missing[cat].nunique() <= 10:
            multi_cats.append(cat)

print()
# Print summary of categorical features
print("================================================================")
print("=============== SUMMARY OF CATEGORICAL FEATURES: ===============")
print()
print(f"binary_cats: {binary_cats}")
print()
print(f"multi_cats: {multi_cats}")
print()

In [ ]:
def get_categorical_features(df, fs):
    # Lists to hold the different types of categoricals
    binary = [] # binary
    multi = [] # multi-level
    non_num = [] # non-numeric

    # get a list of all features
    f_list = df.columns.to_list()

    # get a list of all categorical features
    c_features = list(fs[
                            (fs["attribute"].isin(f_list)) &
                            (fs["type"] == "categorical")
                        ]["attribute"])

    # iterate over each categorical
    for feat in c_features:

        # check if it is any of the ones we don't want to use
        if feat in features_to_drop["categorical"]:
            # continue and do not append to any list
            continue

        # check for binary ones
        if df[feat].nunique() == 2:
            # check if numeric or not
            if df[feat].dtype == "object":
                non_num.append(feat)
            else:
                binary.append(feat)
        # else, it is a multi-level categorical
        else:
            multi.append(feat)

    return binary, multi, non_num

In [ ]:
binary_cats, multi_cats, non_num_cats = get_categorical_features(azdias_low_missing, feature_summary)

print("================================================================")
print("=============== SUMMARY OF CATEGORICAL FEATURES: ===============")
print()
print(f"binary_cats (keep): {binary_cats}")
print()
print(f"multi_cats (keep): {multi_cats}")
print()
print(f"non_num_cats (keep): {non_num_cats}")

#### Discussion 1.2.1: Re-Encode Categorical Features

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding categorical features. Which ones did you keep, which did you drop, and what engineering steps did you perform?)

#### Step 1.2.2: Engineer Mixed-Type Features

There are a handful of features that are marked as "mixed" in the feature summary that require special treatment in order to be included in the analysis. There are two in particular that deserve attention; the handling of the rest are up to your own choices:
- "PRAEGENDE_JUGENDJAHRE" combines information on three dimensions: generation by decade, movement (mainstream vs. avantgarde), and nation (east vs. west). While there aren't enough levels to disentangle east from west, you should create two new variables to capture the other two dimensions: an interval-type variable for decade, and a binary variable for movement.
- "CAMEO_INTL_2015" combines information on two axes: wealth and life stage. Break up the two-digit codes by their 'tens'-place and 'ones'-place digits into two new ordinal variables (which, for the purposes of this project, is equivalent to just treating them as their raw numeric values).
- If you decide to keep or engineer new features around the other mixed-type features, make sure you note your steps in the Discussion section.

Be sure to check `Data_Dictionary.md` for the details needed to finish these tasks.

In [ ]:
# Keep track of the features we have decided to engineer
FEATURES_TO_ENGINEER = ["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"]
FEATURES_TO_ENGINEER

In [ ]:
# Investigate "PRAEGENDE_JUGENDJAHRE" and engineer two new variables.
azdias_low_missing["PRAEGENDE_JUGENDJAHRE"].info()

In [ ]:
data_dictionary["PRAEGENDE_JUGENDJAHRE"]

In [ ]:
# Create mapping dictionaries, using the Data_Dictionary.md as reference
# Map to decades 40s to 90s
decade_map = {
    1: 4, 2: 4, 3: 5, 4: 5, 5: 6, 6: 6, 7: 6, 8: 7, 9: 7, 10: 8, 11: 8, 12: 8, 13: 8, 14: 9, 15: 9
}

# Map to movement
# 1 = Mainstream, 0 = Avantgrade
movement_map = {
    1: 1, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 0, 10: 1, 11: 0, 12: 1, 13: 0, 14: 1, 15: 0
}

In [ ]:
# Investigate "CAMEO_INTL_2015" and engineer two new variables.
# German CAMEO: Wealth / Life Stage Typology, mapped to international code
azdias_low_missing["CAMEO_INTL_2015"].info()

In [ ]:
data_dictionary["CAMEO_INTL_2015"]

In [ ]:
# Since we are separating the digits, rather than mapping, I will attempt to perform
# an operation on each row
temp_df = azdias_low_missing.loc[:10, "CAMEO_INTL_2015"]
print(f"Sample: {temp_df}")
print(f"type(temp): {type(temp_df)}")
print()

print(f"====== SIMULATION OF GETTING DIGITS ======")
for t in temp_df:
    print(f"For: {t}")
    # To get the first digit
    print(f"First Digit: {int(t) // 10}")
    # To get the second digit
    print(f"Second Digit: {int(t) % 10}")
    print()

In [ ]:
# Create the new features
# pop_accepted_tmp["WEALTH"] = pop_accepted_tmp["CAMEO_INTL_2015"].apply(lambda x: int(x) // 10 if not pd.isna(x) else np.nan)
# pop_accepted_tmp["LIFE_STAGE"] = pop_accepted_tmp["CAMEO_INTL_2015"].apply(lambda x: int(x) % 10 if not pd.isna(x) else np.nan)

In [ ]:
# Sanity Check
# pop_accepted_tmp.loc[:, ["CAMEO_INTL_2015", "WEALTH", "LIFE_STAGE"]].head(10)

**IMPORTANT NOTE**: I am only doing analysis for feature selection and re-encoding in Step 1.2. Later on, in Step 1.3 is where I actually perform the preprocessing and transformation. This is because I decided to use custom transformers and pipelines. Separating it all into it's own step is cleaner.

In [ ]:
# Keep track of the new features that will be created
new_features = {
    BINARY: ["MOVEMENT"],
    INTERVAL: ["DECADE"],
    ORDINAL: ["WEALTH", "LIFE_STAGE"]
}

#### Discussion 1.2.2: Engineer Mixed-Type Features

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding mixed-value features. Which ones did you keep, which did you drop, and what engineering steps did you perform?)

#### Step 1.2.3: Complete Feature Selection

In order to finish this step up, you need to make sure that your data frame now only has the columns that you want to keep. To summarize, the dataframe should consist of the following:
- All numeric, interval, and ordinal type columns from the original dataset.
- Binary categorical features (all numerically-encoded).
- Engineered features from other multi-level categorical features and mixed features.

Make sure that for any new columns that you have engineered, that you've excluded the original columns from the final dataset. Otherwise, their values will interfere with the analysis later on the project. For example, you should not keep "PRAEGENDE_JUGENDJAHRE", since its values won't be useful for the algorithm: only the values derived from it in the engineered features you created should be retained. As a reminder, your data should only be from **the subset with few or no missing values**.

In [ ]:
azdias_low_missing.head()

In [ ]:
feature_summary.head()

In [ ]:
azdias_low_missing.columns.to_list()

In [ ]:
def get_numeric_features(df, fs):
    feats = df.columns.to_list()
    return list(fs[
                    (fs["attribute"].isin(feats)) &
                    (fs["type"] == "numeric")]["attribute"])

def get_ordinal_features(df, fs):
    data_cols = df.columns.to_list()
    return list(fs[
                    (fs["attribute"].isin(data_cols)) &
                    (fs["type"] == "ordinal")]["attribute"])

def get_features_to_drop(dropped_feats):
    to_drop = []

    for k, v in dropped_feats.items():
        to_drop.extend(v)

    return to_drop

In [ ]:
def get_feature_selection(df, fs, to_drop):
    bin_cats, mul_cats, no_num_cats = get_categorical_features(azdias_low_missing, fs)
    num_feats = get_numeric_features(df, fs)
    ord_feats = get_ordinal_features(df, fs)

    # Account for the new features:
    bin_cats.extend(new_features[BINARY])
    interval_feats = new_features[INTERVAL]
    ord_feats.extend(new_features[ORDINAL])

    feats_to_keep = ord_feats + num_feats + bin_cats + mul_cats + no_num_cats + interval_feats
    feats_to_drop = get_features_to_drop(to_drop)

    feature_selection = {
        BINARY: bin_cats,
        MULTI: mul_cats,
        NON_NUM: no_num_cats,
        NUMERICAL: num_feats,
        ORDINAL: ord_feats,
        INTERVAL: interval_feats,
        TO_ENGINEER: FEATURES_TO_ENGINEER,
        TO_DROP: feats_to_drop,
        TO_KEEP: feats_to_keep,
    }

    return feature_selection

In [ ]:
def print_feature_selection(f_sel):
    print("================================================================")
    print("=============== SUMMARY OF FEATURES: ===============")
    print()
    print(f"ordinal (keep) {len(f_sel[ORDINAL])}: {f_sel[ORDINAL]}")
    print()
    print(f"numeric (keep) {len(f_sel[NUMERICAL])}: {f_sel[NUMERICAL]}")
    print()
    print(f"binary_cats (keep) {len(f_sel[BINARY])}: {f_sel[BINARY]}")
    print()
    print(f"multi_cats (keep) {len(f_sel[MULTI])}: {f_sel[MULTI]}")
    print()
    print(f"non_num_cats (keep) {len(f_sel[NON_NUM])}: {f_sel[NON_NUM]}")
    print()
    print(f"intervals (keep) {len(f_sel[INTERVAL])}: {f_sel[INTERVAL]}")
    print()
    print(f"features to engineer {len(f_sel[TO_ENGINEER])}: {f_sel[TO_ENGINEER]}")
    print()
    print(f"features that will be created: {new_features}")
    print()
    print(f"Total number of features that will be used: {len(f_sel[TO_KEEP])}")
    print()
    print(f"Features that will be dropped: {f_sel[TO_DROP]}")
    print()
    print(f"Total number of features that will be dropped: {len(f_sel[TO_DROP])}")

feature_selection = get_feature_selection(azdias_low_missing, feature_summary, features_to_drop)
print_feature_selection(feature_selection)

In [ ]:
# https://medium.com/@benjybo7/3-steps-to-implement-a-custom-scikit-learn-transformer-88b5acf3c80f
from sklearn import set_config
set_config(transform_output="pandas")

class CustomEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns_ = None

    def fit(self, X, y=None):
        # Store column names if X is a DataFrame
        self.columns_ = getattr(X, "columns", None)
        return self

    def transform(self, X, y=None):
        # Make a copy to avoid modifying the original
        X_out = X.copy()

        map_to_decade = {1: 4, 2: 4, 3: 5, 4: 5, 5: 6, 6: 6, 7: 6, 8: 7, 9: 7,
                         10: 8, 11: 8, 12: 8, 13: 8, 14: 9, 15: 9}

        map_to_movement = {1: 1, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 0,
                           10: 1, 11: 0, 12: 1, 13: 0, 14: 1, 15: 0}

        # Create new columns
        X_out["DECADE"] = X_out["PRAEGENDE_JUGENDJAHRE"].map(map_to_decade)
        X_out["MOVEMENT"] = X_out["PRAEGENDE_JUGENDJAHRE"].map(map_to_movement)
        X_out["WEALTH"] = X_out["CAMEO_INTL_2015"].apply(
            lambda x: int(x) // 10 if not pd.isna(x) else np.nan
        )
        X_out["LIFE_STAGE"] = X_out["CAMEO_INTL_2015"].apply(
            lambda x: int(x) % 10 if not pd.isna(x) else np.nan
        )

        # Drop original columns
        X_out = X_out.drop(columns=["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"])

        # Re-encode OST_WEST_KZ
        X_out["OST_WEST_KZ"] = X_out["OST_WEST_KZ"].map({'O': 0, 'W': 1})

        return X_out

    def get_feature_names_out(self, input_features=None):

        if input_features is None:
            input_features = self.columns_

        # Build output column names
        output_cols = input_features + ["DECADE", "MOVEMENT", "WEALTH", "LIFE_STAGE"]

        return np.asarray(output_cols, dtype=object)

In [ ]:
def drop_features(X, f_list):
    X_out = X.copy()

    for f in f_list:
        try:
            X_out.drop(columns=f, inplace=True)
        except KeyError:
            print(f"Could not drop column: {f}")

    return X_out

In [ ]:
feature_selection

In [ ]:
# Check multi-categorical columns that we will impute
feature_selection[MULTI]


In [ ]:
# # https://www.geeksforgeeks.org/data-science/custom-transformers-in-scikit-learn-pipelines/
# # https://www.geeksforgeeks.org/machine-learning/ml-one-hot-encoding/
#
# custom_encoder = ColumnTransformer(
#     transformers=[
#         ('custom_encoder', CustomEncoder(), feature_selection[TO_CUSTOM_ENCODE]),
#     ],
#     remainder="passthrough",
#     verbose_feature_names_out=False
# )
#
# reengineering_pipeline = Pipeline([
#     ('custom_encoder', custom_encoder),
#     ('imputer', SimpleImputer(strategy='most_frequent'))
# ])
#
# categorical_preprocessor = ColumnTransformer(
#     transformers=[
#         ('multi_cats', Pipeline([
#             ('imputer', SimpleImputer(strategy='most_frequent')),
#             ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
#         ]), feature_selection[MULTI])
#     ],
#     remainder="passthrough",
#     verbose_feature_names_out=False
# )
#
# imputer = ColumnTransformer([
#     ('impute_multi_cats', SimpleImputer(strategy='most_frequent'), feature_selection[MULTI]),
#     ('impute_numeric', SimpleImputer(strategy='median'), feature_selection[NUMERICAL]),
#     ('impute_binary', SimpleImputer(strategy='most_frequent'), feature_selection[BINARY]),
#     ('impute_ordinals', SimpleImputer(strategy='median'), feature_selection[ORDINAL])
# ], remainder='passthrough', verbose_feature_names_out=False)
#
# prep_pipe = Pipeline([
#     ('replace', replacer_transformer),
#     ('drop_cols', na_col_dropper),
#     ('drop_rows', na_row_dropper),
#     ('re-engineer', reengineering_pipeline),
#     ('imputer', imputer),
#     ('encode', categorical_preprocessor),
#     ('drop_list', FunctionTransformer(drop_features, kw_args={"f_list": feature_selection[TO_DROP]}, validate=False))
# ]).set_output(transform="pandas")

### Step 1.3: Complete Feature Selection: Building Processing and Feature Extraction Pipelines:

For this project, I did a lot of research so that I could do as most a proper job as possible. I stumbled across FeatureTransformers, ColumnTransformers and Pipelines, which just makes it much more robust to preprocess data. So I have decided to implement this in the project.

We are basically preprocessing our data by performing the following steps:
1. Converting Missing Value Codes to NaNs
2. Dropping Columns With Many Missing Values
2. Dropping Rows with Many Missing Values
2. Re-encoding Categorical Features
3. Engineering Mixed Features


**IMPORTANT NOTE**: As per the sklearn documentation, and because later on in this project we will be reversing our transformations, imputations, etc., we will need to make sure all our custom estimators (as sklearn calls them - for this project, they are more specifically transformers) support ```inverse_transform```.
- https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html?utm_source=chatgpt.com#sklearn.pipeline.Pipeline.inverse_transform

#### Step 1.3.1: Converting Missing Value Codes to NaNs:

We already did some testing for this in Step 1.1.1 So here we will just recap.

In [ ]:
# NaN values on raw data
print(f"NaN values on raw data: {azidas_raw.isna().sum().sum()}")

# NaN values on converted data
print(f"NaN values on converted data: {azidas_nan.isna().sum().sum()}")

This was done using a FunctionTransformer:

In [ ]:
replacer_transformer

#### Step 1.3.2: Dropping Columns With Many Missing Values:

We already did some testing for this in Step 1.1.2 So here we will just recap: we dropped columns with a threshold of more than 20% of missing values.

In [ ]:
# Get the missing per columns as a percentage
col_missing_counts = azidas_cols_clean.isna().sum()
missing_pct = (col_missing_counts / len(azidas_cols_clean)) * 100

# If all is good, then there shouldn't be any
assert len(missing_pct[missing_pct > 20]) == 0

And to accomplish this, I created my own custom transformer:

In [ ]:
# We have defined a custom transformer for removing columns with missing data
na_col_dropper

And to accomplish this, I reused my own custom transformer:

In [ ]:
na_row_dropper

#### Step 1.3.3: Dropping Rows with Many Missing Values:

We already did some testing for this in Step 1.1.2 So here we will just recap: we removed rows that had more than 20 missing values.

In [ ]:
# Get Missing Values Per Row
na_per_row = get_na_per_row(azdias_low_missing)

# Make sure no row has more than 20 missing values
assert len(na_per_row[na_per_row > 20]) == 0

#### Step 1.3.4: Re-encoding Categorical Features:

In Step 1.2.1 we determined the categorical features that we will keep and drop. We decided to keep 11 features and drop 7 To recap:

In [ ]:
# Get the categorical features, by type, and print
b, m, no = get_categorical_features(azdias_low_missing, feature_summary)
print(f"Binary Categorical Features: {b}\n\n"
      f"Multi-Level Categorical Features: {m}\n\n"
      f"Non-numerical Categorical Features: {no}\n")

# Print the categoricals to drop
print(f"Categorical Features to Drop: {features_to_drop["categorical"]}")

Now, we will perform the following steps for preprocessing and transforming the features we will **keep**:

1. **Binary Categoricals**: Keep as is.
2. **Multi-Level Categoricals**: Encode using one hot encode.
3. **Non-numerical Categoricals**: Only one, 'OST_WEST_KZ', we re-encode ('O', 'W') to (0, 1)

#### Step 1.3.4.1: Encode Multi-Level Categoricals:
We will encode the multi-level categoricals using one-hot encoding. For this, we will use a ColumnTransformer, which makes it easier to perform transformations on specific columns.

References:
- https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
- https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_pipeline_display.html

In [ ]:
# Create the OneHotEncoder, we use handle_unknown='ignore' to handle missing values
# and sparse_ouput=False, because Pandas does not support sparse_output
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_encoder

Now we test the One Hot Encoder. Please refer to the "feature_selection.xlsx" file, which I created to help me make sure I am doing the proper feature selection. Based on the columns, unique types, etc., we should get the following after encoding the 8 feature columns:
- **CJT_GESAMTTYP**: We should get 6 new columns + 1 NaN column
- **FINANZTYP**: We should get 6 new columns
- **GFK_URLAUBERTYP**: We should get 12 new columns + 1 NaN column
- **LP_FAMILIE_GROB**: We should get 5 new columns + 1 NaN column
- **LP_STATUS_GROB**: We should get 5 new columns + 1 NaN column
- **SHOPPER_TYP**: We should get 4 new columns + 1 NaN column
- **ZABEOTYP**: We should get 6 new columns
- **CAMEO_DEUG_2015**: We should get 9 new columns + 1 NaN column

In total, this adds up to 59 new columns

In [ ]:
# To test, make sure we will only use the Multi-Level Categoricals:
azdias_low_missing[feature_selection[MULTI]].columns.to_list()

In [ ]:
# Transform the Multi-Level Categoricals
test_df = categorical_encoder.fit_transform(azdias_low_missing[feature_selection[MULTI]])

# Get the column names (for when we inverse_transform)
original_columns = azdias_low_missing[feature_selection[MULTI]].columns.to_list()
print(f"Columns: {original_columns}")

# View Results
test_df.head()

As we can see, we indeed get 59 features. Now let us try to do the inverse transform:

In [ ]:
# Perform the inverse transform
test_df = categorical_encoder.inverse_transform(test_df)

# We get a numpy array, so we convert to DataFrame
test_df = pd.DataFrame(test_df, columns=original_columns)

# View Results
test_df.head()

As can be seen, we were able to OneHotEncode our multi-level categoricals and also do the inverse transform, while preserving the column names. Later, when we build our ColumnTransformers and Pipelines, we will use some parameters so that we get back a pandas DataFrame (when transforming and inverse transforming) and the feature names are preserved.

#### Step 1.3.4.1: Re-Encoding Non-numerical Categoricals

As per the instructions, we will re-encode 'OST_WEST_KZ'. To do this, I will use a custom FunctionTransformer, so that we can use it in a ColumnTransform and in a Pipeline.

References:
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html

In [ ]:
# Make sure we are only testing on the relevant column
# TODO: Just noticed, fix variable name for the azdias dataframes
azdias_low_missing[feature_selection[NON_NUM]].columns.to_list()

In [ ]:
# Define the FunctionTransformer
# We use a lambda function to do the re-encoding
# And for fun, another one for the inverse_transform

non_num_transformer = FunctionTransformer(
    func=lambda X: X["OST_WEST_KZ"].map({'O': 0, 'W': 1}),
    inverse_func=lambda X: X["OST_WEST_KZ"].map({0: 'O', 1: 'W'}),
    # Prevent an error and output feature names will be equal to the input feature names
    feature_names_out="one-to-one",
    check_inverse=False
)
non_num_transformer

In [ ]:
# Transform the feature
test_df = non_num_transformer.fit_transform(azdias_low_missing[feature_selection[NON_NUM]])

# View Results
test_df.head()

As we can see, the custom Function Transformer seems to work. Now we check:

In [ ]:
# Sanity Check:
print(f"Original Value Counts: {azdias_low_missing[feature_selection[NON_NUM]].value_counts()}")
print(f"Re-Encoded Value Counts: {test_df.value_counts()}")

# Make sure that both dataframes are the same shape
assert azdias_low_missing[feature_selection[NON_NUM]].shape == test_df.shape

# Assert they only have 2 unique values
assert len(azdias_low_missing[feature_selection[NON_NUM]].value_counts().unique()) == 2
assert len(test_df.value_counts().unique()) == 2

# Iterate and check 5000 random samples/rows
for _ in range(5000):
    # Get a random index of a row
    i = np.random.randint(azdias_low_missing.shape[0])

    # We can compare values since we know for sure both dataframes only have 2 unique values
    comp_1 = 1 if azdias_low_missing[feature_selection[NON_NUM]].iloc[i, :].item() == 'W' else 0
    comp_2 = test_df.iloc[i, :].item()
    assert comp_1 == comp_2

As we can see, we have successfully Re-Encoded 'OST_WEST_KZ'. Now let us check the inverse transform (for fun).

In [ ]:
test_df = non_num_transformer.inverse_transform(test_df)
test_df.head()

As can be seen, we were able to Re-Encode 'OST_WEST_KZ' and also do the inverse transform, while preserving the column names. Later, when we build our ColumnTransformers and Pipelines, we will use some parameters so that we get back a pandas DataFrame (when transforming and inverse transforming) and the feature names are preserved.

#### Step 1.3.5: Engineering Mixed Features:
We are reengineering columns: ["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"]. After encoding, we should obtain the columns:

For "PRAEGENDE_JUGENDJAHRE":
- **DECADE**: Interval
- **MOVEMENT**: Binary

For "CAMEO_INTL_2015":
- **WEALTH**: Ordinal
- **LIFE_STAGE**: Ordinal

To take advantage of the tools sklearn has available for us, I will be using:
- A custom encoder: implementing a custom encoder so that we can add it to a pipeline. I could have used a FunctionTransformer, but a custom transformer allows us to implement get_features_names_out(). This is very important because we are **manually** creating 4 columns. With this method, we can include our custom encoder/transformer in a pipeline and propagate the correct columns down the pipeline.


In [ ]:
# https://medium.com/@benjybo7/3-steps-to-implement-a-custom-scikit-learn-transformer-88b5acf3c80f
from sklearn import set_config
set_config(transform_output="pandas")

class MixedFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns_ = None

    def fit(self, X, y=None):
        # Store column names if X is a DataFrame
        self._original_columns = ["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"]
        return self

    def transform(self, X, y=None):
        # Make a copy to avoid modifying the original
        X_out = X.copy()

        map_to_decade = {1: 4, 2: 4, 3: 5, 4: 5, 5: 6, 6: 6, 7: 6, 8: 7, 9: 7,
                         10: 8, 11: 8, 12: 8, 13: 8, 14: 9, 15: 9}

        map_to_movement = {1: 1, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 0,
                           10: 1, 11: 0, 12: 1, 13: 0, 14: 1, 15: 0}

        # Create new columns
        X_out["DECADE"] = X_out["PRAEGENDE_JUGENDJAHRE"].map(map_to_decade)
        X_out["MOVEMENT"] = X_out["PRAEGENDE_JUGENDJAHRE"].map(map_to_movement)
        X_out["WEALTH"] = X_out["CAMEO_INTL_2015"].apply(
            lambda x: int(x) // 10 if not pd.isna(x) else np.nan
        )
        X_out["LIFE_STAGE"] = X_out["CAMEO_INTL_2015"].apply(
            lambda x: int(x) % 10 if not pd.isna(x) else np.nan
        )

        # Drop original columns
        X_out = X_out.drop(columns=["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"])

        return X_out

    def get_feature_names_out(self, input_features=None):

        if input_features is None:
            input_features = self.columns_

        # Build output column names
        output_cols = input_features + ["DECADE", "MOVEMENT", "WEALTH", "LIFE_STAGE"]

        return output_cols

    def inverse_transform(self, X, y=None):
        # In this case, for analysis, we do not want to inverse transform
        # So we do nothing
        return X

mixed_feature_engineer = MixedFeatureEngineer()
mixed_feature_engineer

Now I will test my custom engineer transformer.

In [ ]:
# Check that we will operate on the right columns
azdias_low_missing[feature_selection[TO_ENGINEER]].columns.to_list()

Since we are working with a pipeline, we need to impute first (to remove NaN values) and then encode, if not, we will get a column for NaN, which can affect our results. We use the custom pipeline, since it imputes and then uses our custom encoder.
- Reference: https://medium.com/@prasanth32888/customer-segmentation-using-k-means-clustering-fe73bee85c57 (Part 2. Step 1: Handling Missing Data)

In [ ]:
# Preprocess the data. We use fit_transform, as the custom pipeline will call fit_transform. Fit shouldn't do anything.
test_df = mixed_feature_engineer.fit_transform(azdias_low_missing[feature_selection[TO_ENGINEER]])

# Show Results
test_df.head()

As we can see, the custom MixedFeatureEngineer seems to work. Now we check:

In [ ]:
# Sanity Check
# Make sure that both dataframes have same number of rows
assert azdias_low_missing[feature_selection[TO_ENGINEER]].shape[0] == test_df.shape[0]

# Make sure we created 4 features
assert test_df.shape[1] == 4

# And we visualize to compare PRAEGENDE_JUGENDJAHRE
comp_df = pd.concat([azdias_low_missing[["PRAEGENDE_JUGENDJAHRE"]], test_df[["DECADE", "MOVEMENT"]]], axis=1)

print(f"Decade Map: {decade_map}")
print()
print(f"Movement Map: {movement_map}")

# We sample 100 rows and visually check
comp_df.sample(100)

In [ ]:
# And we visualize to compare CAMEO_INTL_2015
comp_df = pd.concat([azdias_low_missing[["CAMEO_INTL_2015"]], test_df[["WEALTH", "LIFE_STAGE"]]], axis=1)

# We sample 100 rows and visually check
comp_df.sample(100)

And lastly, we check the inverse transform:

In [ ]:
# Perform the inverse transform
test_df = mixed_feature_engineer.inverse_transform(test_df)

# Check that nothing happened after applying inverse transform
test_df.head()

#### Step 1.3 Conclusion:

As was seen in this step, we were able to successfully Select and Re-Encode Features. The next step will be to build our Pipelines.

### Step 1.4: Build Processing Pipelines

As per the sklearn documentation, Pipelines serve multiple purposes:

- **Convenience and Encapsulation**: only call fit and predict once the data is fit to a whole sequence of estimators (in my case transfomers)
- **Safety**: Pipelines avoid data leakage. The same samples are used to train the transformers

For this reason, although not required in this project, I decided to implement Pipelines so that I could have a more robust and reusable project, which I can reference in the future.

References:
- https://scikit-learn.org/stable/modules/compose.html#pipeline
- https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
- https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_pipeline_display.html
- https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html

#### Step 1.4: Build Pipeline

Column transformers allow us to conveniently apply transforms to columns of a pandas Data Frame

Let us recall the order:
1. Converting Missing Value Codes to NaNs
2. Dropping Columns With Many Missing Values
3. Dropping Rows with Many Missing Values
4. Re-encoding Categorical Features
5. Engineering Mixed Features

And we have already created our transfomers:
1. ```replacer_transfomer```
2. ```na_col_dropper```
3. ```na_row_dropper```
4. ```non_num_transformer```
5. ```mixed_feature_engineer```


All we have to do now is combine ColumnTransformers and Pipelines to preprocess our data. Only those transformers that need specific columns will use a ColumnTransformer. As per the sklearn documentation the transformers parameter of a : "List of (name, transformer, columns) tuples specifying the transformer objects to be applied to subsets of the data."

**IMPORTANT NOTE**: I will be using ColumnTransformers and Pipelines on both the general population demographics and the customer data demographics. One thing to note is that we cannot be assured that the same transformations will be applied to both datasets. For instance, the customer dataset not only has different number of rows, but the number of missing values can differ per column, when compared to the general population data, so rather than perform the same step for dropping columns at a threshold, it makes more sense to drop the same columns as the general population dataset. This is why I kept track of the dropped columns in a dictionary called ```features_to_drop```. So we will do this in two steps, one for each: 1) General Population Demographics and 2) Customer Data Demographics. That way, we can use the clean function on BOTH.

#### Step 1.4.1: General Population Preprocessing:

Recall that we created a ```feature_selection``` dictionary to summarize all the final features:

In [ ]:
pprint(feature_selection, depth=1)

For removing missings or unknowns and dropping rows and columns, we use a Pipeline, without a classifier. As per the sklearn documentation: Pipeline can be used to chain multiple estimators into one. This is useful as there is often a fixed sequence of steps in processing the data.

**IMPORTANT NOTE**: We also have to drop those columns that we manually chose top drop. These are found in: ```feature_selection['features_to_drop']```. For this, we can use a simple FunctionTransfomer:

In [ ]:
# Recall the features to drop
pprint(feature_selection[TO_DROP])

In [ ]:
def drop_features(X, f_list):
    X_out = X.copy()

    for f in f_list:
        if f not in X.columns:
            continue

        print(f"Attempting to drop: {f}")

        try:
            X_out = X_out.drop(columns=f)
        except KeyError:
            print(f"Could not drop: {f}")
    return X_out

drop_features_transfomer = FunctionTransformer(
    drop_features,
    kw_args={
        "f_list": feature_selection[TO_DROP],
    },
    validate=False
)

In [ ]:
# Define our Pipeline
population_cleaning_pipeline = Pipeline(
    steps=[
        ('drop_list', drop_features_transfomer),
        ('replace_nan', replacer_transformer),
        ('drop_na_cols', na_col_dropper),
        ('drop_na_rows', na_row_dropper),
    ]
)

For all the feature transformations, re-encoding, encoding, engineering, we will use a ColumnTransformer. We will refer to the entire process as 'feature_engineering'.

In [ ]:
# Define our ColumnTransformer
feature_engineering = ColumnTransformer(
    transformers= [
        ('categorical_encoder', categorical_encoder, feature_selection[MULTI]),
        ('non_num_transformer', non_num_transformer, feature_selection[NON_NUM]),
        ('engineer_mixed', mixed_feature_engineer, feature_selection[TO_ENGINEER])
    ],
    # By default, columns not transformed are drop.
    # We set remainder='passthrough', we ensure
    # those columns are passed through (without transformation)
    # and are not dropped.
    remainder='passthrough',
    # We do not want to prefix anything to our feature names
    verbose_feature_names_out=False
)

Now we combine these both, to complete the preprocessing for population demographic data:

In [ ]:
population_preprocessing = Pipeline(
    steps=[
        ('clean_data', population_cleaning_pipeline),
        ('enginner_features', feature_engineering)
    ]
).set_output(transform="pandas")

And now, we test:

In [ ]:
# Make sure our raw data hasn't changed
azidas_raw.head()

In [ ]:
azdias_clean = None
azdias_clean = population_cleaning_pipeline.fit_transform(azidas_raw)
azdias_clean.head()

In [ ]:
azdias_eng = None
azdias_eng = feature_engineering.fit_transform(azdias_clean)
azdias_eng.head()

In [ ]:
set(azdias_eng.columns.to_list()).difference(set(azdias_clean.columns.to_list()))

In [ ]:
azdias_engineered = None
azdias_engineered = population_preprocessing.fit_transform(azidas_raw)
azdias_engineered.head()

In [ ]:
print(f"Results: {len(azdias_engineered.columns)}")
azdias_engineered.columns.to_list()

In [ ]:
azdias_engineered = drop_features(azdias_engineered, feature_selection[TO_DROP])
azdias_engineered.head()

In [ ]:
def clean_data(df, fs, is_customer_data=False, columns_to_drop=None):
    """
    Perform feature trimming, re-encoding, and engineering for demographics
    data
    
    INPUT: Demographics DataFrame
    OUTPUT: Trimmed and cleaned demographics DataFrame
    """

    ft_cats = get_feature_selection(df, fs)
    fts_to_encode = ft_cats[NON_NUM] + ft_cats[ENGINEERED]
    print_feature_selection(ft_cats)

    imputer_f = ColumnTransformer([
        ('impute_multi_cats', SimpleImputer(strategy='most_frequent'), ft_cats[MULTI]),
        ('impute_encode_cols', SimpleImputer(strategy='most_frequent'), fts_to_encode),
        ('impute_numeric', SimpleImputer(strategy='median'), ft_cats[NUMERICAL]),
        ('impute_binary', SimpleImputer(strategy='most_frequent'), ft_cats[BINARY]),
        ('impute_ordinals', SimpleImputer(strategy='median'), ft_cats[ORDINAL])
    ], remainder='passthrough', verbose_feature_names_out=False)

    encoder_f = ColumnTransformer(
        transformers=[
            ('multi_cat', OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop=None), ft_cats[MULTI]),
            ('encode', CustomEncoder(), fts_to_encode)
        ],
        remainder="passthrough",
        verbose_feature_names_out=False
    )

    if not is_customer_data:

        preprocess_pipeline = Pipeline([
            ('replace', transformers["replacer"]),
            ('drop_cols', transformers["drop_na_cols"]),
            ('drop_rows', transformers["drop_na_rows"]),
            ('imputer', imputer_f),
            ('encode', encoder_f),
            ('drop_list', FunctionTransformer(drop_features, kw_args={"f_list": ft_cats[TO_DROP]}, validate=False))
        ]).set_output(transform="pandas")

        return preprocess_pipeline.fit_transform(df)

    else:
        if columns_to_drop:
            cust_preprocess_pipeline = Pipeline([
            ('replace', transformers["replacer"]),
            ('imputer', imputer_f),
            ('encode', encoder_f),
            ('drop_list', FunctionTransformer(drop_features, kw_args={"f_list": columns_to_drop}, validate=False))
        ]).set_output(transform="pandas")

        return cust_preprocess_pipeline.fit_transform(df)


In [ ]:
pop_data_df, feat_sum = load_data()
pop_data_df.head()

In [ ]:
feat_sum.head()

In [ ]:
pop_data_df = clean_data(pop_data_df, feat_sum)

In [ ]:
pop_data_df.head()

## Step 2: Feature Transformation

### Step 2.1: Apply Feature Scaling

Before we apply dimensionality reduction techniques to the data, we need to perform feature scaling so that the principal component vectors are not influenced by the natural differences in scale for features. Starting from this part of the project, you'll want to keep an eye on the [API reference page for sklearn](http://scikit-learn.org/stable/modules/classes.html) to help you navigate to all of the classes and functions that you'll need. In this substep, you'll need to check the following:

- sklearn requires that data not have missing values in order for its estimators to work properly. So, before applying the scaler to your data, make sure that you've cleaned the DataFrame of the remaining missing values. This can be as simple as just removing all data points with missing data, or applying an [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) to replace all missing values. You might also try a more complicated procedure where you temporarily remove missing values in order to compute the scaling parameters before re-introducing those missing values and applying imputation. Think about how much missing data you have and what possible effects each approach might have on your analysis, and justify your decision in the discussion section below.
- For the actual scaling function, a [StandardScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) instance is suggested, scaling each feature to mean 0 and standard deviation 1.
- For these classes, you can make use of the `.fit_transform()` method to both fit a procedure to the data as well as apply the transformation to the data at the same time. Don't forget to keep the fit sklearn objects handy, since you'll be applying them to the customer demographics data towards the end of the project.

In [ ]:
# If you've not yet cleaned the dataset of all NaN values, then investigate and
# do that now.
pop_data_df.isna().sum().sum()

In [ ]:
numeric_features = get_numeric_features(pop_data_df, feat_sum)
ordinal_features = get_ordinal_features(pop_data_df, feat_sum)

scaler = ColumnTransformer([
    ('scale', StandardScaler(), numeric_features + ordinal_features)
], remainder='passthrough', verbose_feature_names_out=False)

In [ ]:
pop_data_df = scaler.fit_transform(pop_data_df)

### Discussion 2.1: Apply Feature Scaling

(Double-click this cell and replace this text with your own text, reporting your decisions regarding feature scaling.)

### Step 2.2: Perform Dimensionality Reduction

On your scaled data, you are now ready to apply dimensionality reduction techniques.

- Use sklearn's [PCA](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) class to apply principal component analysis on the data, thus finding the vectors of maximal variance in the data. To start, you should not set any parameters (so all components are computed) or set a number of components that is at least half the number of features (so there's enough features to see the general trend in variability).
- Check out the ratio of variance explained by each principal component as well as the cumulative variance explained. Try plotting the cumulative or sequential values using matplotlib's [`plot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.plot.html) function. Based on what you find, select a value for the number of transformed features you'll retain for the clustering part of the project.
- Once you've made a choice for the number of components to keep, make sure you re-fit a PCA instance to perform the decided-on transformation.

In [ ]:
# Apply PCA to the data.
n_comps = int(pop_data_df.shape[1] / 2)
print(f"Number of components: {n_comps}")
# pca = PCA(n_components=n_comps)
pca = PCA()
X_pca = pca.fit_transform(pop_data_df)

In [ ]:
# Investigate the variance accounted for by each principal component.
print(pca.explained_variance_ratio_.shape)
pca.explained_variance_ratio_

In [ ]:
n_components = len(pca.explained_variance_ratio_)
ind = np.arange(n_components)
vals = pca.explained_variance_ratio_

plt.figure(figsize=(10, 6))
ax = plt.subplot(111)

c_vals = np.cumsum(vals)

ax.bar(ind, vals)
ax.plot(ind, c_vals)

# ax.xaxis.set_tick_params(width=0)
# ax.yaxis.set_tick_params(width=2, length=12)

ax.set_xlabel("Principal Component")
ax.set_ylabel("Variance-Explained Ratio")
plt.title('Explained Variance Per Principal Component')
plt.show()

In [ ]:
# Determine number of components to reach 90% variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
cumulative_variance

In [ ]:
for n in range(len(cumulative_variance)):
    if cumulative_variance[n] * 100 >= 95:
        print(f"Number of components to reach at least 90% variance: {n}")
        break

In [ ]:
# Re-apply PCA to the data while selecting for number of components to retain.
pca_f = PCA(n_components=58)
X_pca = pca_f.fit_transform(pop_data_df)

In [ ]:
X_pca.head()

In [ ]:
X_pca.shape

### Discussion 2.2: Perform Dimensionality Reduction

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding dimensionality reduction. How many principal components / transformed features are you retaining for the next step of the analysis?)

### Step 2.3: Interpret Principal Components

Now that we have our transformed principal components, it's a nice idea to check out the weight of each variable on the first few components to see if they can be interpreted in some fashion.

As a reminder, each principal component is a unit vector that points in the direction of highest variance (after accounting for the variance captured by earlier principal components). The further a weight is from zero, the more the principal component is in the direction of the corresponding feature. If two features have large weights of the same sign (both positive or both negative), then increases in one tend expect to be associated with increases in the other. To contrast, features with different signs can be expected to show a negative correlation: increases in one variable should result in a decrease in the other.

- To investigate the features, you should map each weight to their corresponding feature name, then sort the features according to weight. The most interesting features for each principal component, then, will be those at the beginning and end of the sorted list. Use the data dictionary document to help you understand these most prominent features, their relationships, and what a positive or negative value on the principal component might indicate.
- You should investigate and interpret feature associations from the first three principal components in this substep. To help facilitate this, you should write a function that you can call at any time to print the sorted list of feature weights, for the *i*-th principal component. This might come in handy in the next step of the project, when you interpret the tendencies of the discovered clusters.

In [ ]:
print(pca_f.components_.shape)
pca_f.components_

In [ ]:
# Map weights for the first principal component to corresponding feature names
# and then print the linked values, sorted by weight.
# HINT: Try defining a function here or in a new cell that you can reuse in the
# other cells.
def map_component_weights(pca, df, pc_no):
    feature_names = df.columns
    weights = pca.components_[pc_no]

    assert len(feature_names) == len(weights)

    # weight_mapping = {
    #     n : w
    #     for n, w in zip(feature_names, weights)
    # }

    # weight_mapping = sorted(weight_mapping.items(), key=lambda x: x[1], reverse=True)

    for n, w in sorted(zip(feature_names, weights), key=lambda x: x[1], reverse=True):

        if n in ["WEALTH", "LIFE_STAGE"]:
            idx = "CAMEO_INTL_2015"
        elif n in ["DECADE", "MOVEMENT"]:
            idx = "PRAEGENDE_JUGENDJAHRE"
        else:
            idx = n
        try:
            print(f"Feature Name: {n:<20} | {data_dictionary[idx]["desc"][:50]:<50} | Weight: {w:>8.4f}")
        except KeyError:
            print(f"Feature Name: {n:<20} | {" ":<50} | Weight: {w:>8.4f}")

In [ ]:
for k, v  in data_dictionary.items():
    if v['type'] == "mixed":
        print(k)

In [ ]:
map_component_weights(pca_f, pop_data_df, pc_no=0)

In [ ]:
# Map weights for the second principal component to corresponding feature names
# and then print the linked values, sorted by weight.
map_component_weights(pca_f, pop_data_df, pc_no=1)

In [ ]:
# Map weights for the third principal component to corresponding feature names
# and then print the linked values, sorted by weight.
map_component_weights(pca_f, pop_data_df, pc_no=2)

### Discussion 2.3: Interpret Principal Components

(Double-click this cell and replace this text with your own text, reporting your observations from detailed investigation of the first few principal components generated. Can we interpret positive and negative values from them in a meaningful way?)

## Step 3: Clustering

### Step 3.1: Apply Clustering to General Population

You've assessed and cleaned the demographics data, then scaled and transformed them. Now, it's time to see how the data clusters in the principal components space. In this substep, you will apply k-means clustering to the dataset and use the average within-cluster distances from each point to their assigned cluster's centroid to decide on a number of clusters to keep.

- Use sklearn's [KMeans](http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans) class to perform k-means clustering on the PCA-transformed data.
- Then, compute the average difference from each point to its assigned cluster's center. **Hint**: The KMeans object's `.score()` method might be useful here, but note that in sklearn, scores tend to be defined so that larger is better. Try applying it to a small, toy dataset, or use an internet search to help your understanding.
- Perform the above two steps for a number of different cluster counts. You can then see how the average distance decreases with an increasing number of clusters. However, each additional cluster provides a smaller net benefit. Use this fact to select a final number of clusters in which to group the data. **Warning**: because of the large size of the dataset, it can take a long time for the algorithm to resolve. The more clusters to fit, the longer the algorithm will take. You should test for cluster counts through at least 10 clusters to get the full picture, but you shouldn't need to test for a number of clusters above about 30.
- Once you've selected a final number of clusters to use, re-fit a KMeans instance to perform the clustering operation. Make sure that you also obtain the cluster assignments for the general demographics data, since you'll be using them in the final Step 3.3.

In [ ]:
from sklearn.cluster import KMeans
# Over a number of different cluster counts...
models = [KMeans(n_clusters=k).fit(X_pca) for k in np.arange(start=2, stop=30)]

# SSE == WCSS == INERTIA
# SCORE IS THE OPPOSITE OF THE ABOVE
wcss = [m.inertia_ for m in models]
wcss

In [ ]:
print(len(models))

In [ ]:
# Investigate the change in within-cluster distance across number of clusters.
# HINT: Use matplotlib's plot function to visualize this relationship.
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(np.arange(start=2, stop=30), wcss, linestyle='--', marker='o', color='b')
ax.set_xlabel('K')
ax.set_ylabel('WCSS')
ax.set_title('K-Means Clustering Scores')

ax.set_xticks(np.arange(2, 30, 1))
ax.grid(True, which='both', linestyle='--', linewidth=0.6, alpha=0.7)

plt.show()

In [ ]:
#https://www.geeksforgeeks.org/machine-learning/calinski-harabasz-index-cluster-validity-indices-set-3/

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

random_state = 42
silhouette_scores = []
calinski_harabasz_scores = []
davies_bouldin_scores = []


# Silhouette Score
for i in np.arange(0, len(models)):

    labels = models[i].fit_predict(X_pca)

    silhouette_scores.append(silhouette_score(X_pca, labels, sample_size=10000))
    calinski_harabasz_scores.append(calinski_harabasz_score(X_pca, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_pca, labels))

    print("For k_clusters =", i + 2, "The silhouette_score using K-Means is:", silhouette_scores[i],
          "| The Calinski Harabasz Score is:", calinski_harabasz_scores[i],
          "| And the Davies Boulding Score is:", davies_bouldin_scores[i])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

wcss_norm = scaler.fit_transform(np.asarray(wcss).reshape(-1, 1))
silhouette_scores_norm = scaler.fit_transform(np.asarray(silhouette_scores).reshape(-1, 1))
ca_ha_scores_norm = scaler.fit_transform(np.asarray(calinski_harabasz_scores).reshape(-1, 1))
db_scores_norm = scaler.fit_transform(np.asarray(davies_bouldin_scores).reshape(-1, 1))

k_range = np.arange(2, 30)

fig, ax = plt.subplots(figsize=(14, 10))

ax.plot(k_range, wcss_norm, label='Inertia (normalized)', linewidth=2.5, marker='o', markersize=8, color='b')
ax.plot(k_range, silhouette_scores_norm, label='Silhouette Score (normalized)', linewidth=2.5, marker='o', markersize=8, color='r')
ax.plot(k_range, ca_ha_scores_norm, label='Calinski–Harabasz (normalized)', linewidth=2.5, marker='o', markersize=8, color='y')
ax.plot(k_range, db_scores_norm, label='Davies-Bouldin (normalized)', linewidth=2.5, marker='o', markersize=8, color='lime')

ax.set_title("Clustering Metrics Across k", fontsize=20, pad=20)
ax.set_xlabel("Number of Clusters (k)", fontsize=16, labelpad=15)
ax.set_ylabel("Normalized Score (0–1)", fontsize=16, labelpad=15)

ax.set_xticks(np.arange(2, 30, 1))

ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax.tick_params(axis='both', which='major', labelsize=14)

ax.legend(fontsize=14, framealpha=0.9)

plt.show()

In [ ]:
# Re-fit the k-means model with the selected number of clusters and obtain
# cluster predictions for the general population demographics data.
model_f = KMeans(n_clusters=6).fit(X_pca)
pop_labels = model_f.predict(X_pca)
pop_labels

### Discussion 3.1: Apply Clustering to General Population

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding clustering. Into how many clusters have you decided to segment the population?)

### Step 3.2: Apply All Steps to the Customer Data

Now that you have clusters and cluster centers for the general population, it's time to see how the customer data maps on to those clusters. Take care to not confuse this for re-fitting all of the models to the customer data. Instead, you're going to use the fits from the general population to clean, transform, and cluster the customer data. In the last step of the project, you will interpret how the general population fits apply to the customer data.

- Don't forget when loading in the customers data, that it is semicolon (`;`) delimited.
- Apply the same feature wrangling, selection, and engineering steps to the customer demographics using the `clean_data()` function you created earlier. (You can assume that the customer demographics data has similar meaning behind missing data patterns as the general demographics data.)
- Use the sklearn objects from the general demographics data, and apply their transformations to the customers data. That is, you should not be using a `.fit()` or `.fit_transform()` method to re-fit the old objects, nor should you be creating new sklearn objects! Carry the data through the feature scaling, PCA, and clustering steps, obtaining cluster assignments for all of the data in the customer demographics data.

In [ ]:
# Load in the customer demographics data.


### Step 3.3: Compare Customer Data to Demographics Data

At this point, you have clustered data based on demographics of the general population of Germany, and seen how the customer data for a mail-order sales company maps onto those demographic clusters. In this final substep, you will compare the two cluster distributions to see where the strongest customer base for the company is.

Consider the proportion of persons in each cluster for the general population, and the proportions for the customers. If we think the company's customer base to be universal, then the cluster assignment proportions should be fairly similar between the two. If there are only particular segments of the population that are interested in the company's products, then we should see a mismatch from one to the other. If there is a higher proportion of persons in a cluster for the customer data compared to the general population (e.g. 5% of persons are assigned to a cluster for the general population, but 15% of the customer data is closest to that cluster's centroid) then that suggests the people in that cluster to be a target audience for the company. On the other hand, the proportion of the data in a cluster being larger in the general population than the customer data (e.g. only 2% of customers closest to a population centroid that captures 6% of the data) suggests that group of persons to be outside of the target demographics.

Take a look at the following points in this step:

- Compute the proportion of data points in each cluster for the general population and the customer data. Visualizations will be useful here: both for the individual dataset proportions, but also to visualize the ratios in cluster representation between groups. Seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) or [`barplot()`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function could be handy.
  - Recall the analysis you performed in step 1.1.3 of the project, where you separated out certain data points from the dataset if they had more than a specified threshold of missing values. If you found that this group was qualitatively different from the main bulk of the data, you should treat this as an additional data cluster in this analysis. Make sure that you account for the number of data points in this subset, for both the general population and customer datasets, when making your computations!
- Which cluster or clusters are overrepresented in the customer dataset compared to the general population? Select at least one such cluster and infer what kind of people might be represented by that cluster. Use the principal component interpretations from step 2.3 or look at additional components to help you make this inference. Alternatively, you can use the `.inverse_transform()` method of the PCA and StandardScaler objects to transform centroids back to the original data space and interpret the retrieved values directly.
- Perform a similar investigation for the underrepresented clusters. Which cluster or clusters are underrepresented in the customer dataset compared to the general population, and what kinds of people are typified by these clusters?

In [ ]:
# Compare the proportion of data in each cluster for the customer data to the
# proportion of data in each cluster for the general population.


In [ ]:
# Differences in proportions


In [ ]:
# What kinds of people are part of a cluster that is overrepresented in the
# customer data compared to the general population?


In [ ]:
# What kinds of people are part of a cluster that is underrepresented in the
# customer data compared to the general population?


### Discussion 3.3: Compare Customer Data to Demographics Data

(Double-click this cell and replace this text with your own text, reporting findings and conclusions from the clustering analysis. Can we describe segments of the population that are relatively popular with the mail-order company, or relatively unpopular with the company?)

> Congratulations on making it this far in the project! Before you finish, make sure to check through the entire notebook from top to bottom to make sure that your analysis follows a logical flow and all of your findings are documented in **Discussion** cells. Once you've checked over all of your work, you should export the notebook as an HTML document to submit for evaluation. You can do this from the menu, navigating to **File -> Download as -> HTML (.html)**. You will submit both that document and this notebook for your project submission.